# Git Version Control Practice Workbook

This exercise-heavy workbook builds Git habits for the Calgary Spatial ETL project. You will predict, type, run, inspect, and explain. Exercises use completed starter answers marked `TODO` so you can erase and rebuild them during retrieval practice.

## How to use it

1. Read the prompt and predict the result.
2. Edit the TODO answer cell without looking ahead.
3. Run the answer cell, then its separate assertion cell.
4. Read Git output as evidence before changing anything.
5. Repeat the exercise from memory on the review schedule.

Unfinished or deliberately broken debugging cells may fail until you repair them.

## Safety Contract and Verified Starting Point

The actual repository is `/home/kanon/calgary-spatial-etl`, its branch is `main`, and it currently has no commits. **No executable cell in this notebook mutates that repository.** All `init`, `config`, `add`, `commit`, `restore`, `switch`, and `merge` practice happens inside `TemporaryDirectory` sandbox repositories with fake, repository-local identity.

Never add a remote, fetch, pull, push, or contact a network from this notebook. Commands for the actual repository are shown later only as a manual checklist.

Safety reminders:

- Never put access tokens in a notebook. Use browser/credential-manager authentication or SSH.
- If a secret is committed, deleting the file is insufficient: rotate the secret immediately and remediate Git history.
- `git reset --hard` discards working-tree and index changes. Treat it as destructive and do not use it here.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
import os
import subprocess

ACTUAL_REPO = Path("/home/kanon/calgary-spatial-etl")  # Documentation only; never pass this to git_run.


def git_run(repo: Path, *args: str, check: bool = True) -> subprocess.CompletedProcess[str]:
    """Run an offline Git command in an explicit temporary sandbox."""
    resolved = repo.resolve()
    temp_root = Path(os.environ.get("TMPDIR", "/tmp")).resolve()
    if temp_root not in resolved.parents:
        raise ValueError(f"Refusing Git mutation outside a temporary sandbox: {resolved}")
    blocked = {"remote", "fetch", "pull", "push", "clone", "ls-remote"}
    if args and args[0] in blocked:
        raise ValueError(f"Network/remote command blocked in this workbook: {args[0]}")
    return subprocess.run(
        ["git", *args], cwd=resolved, check=check, text=True,
        capture_output=True,
    )


def configure_sandbox(repo: Path) -> None:
    git_run(repo, "config", "--local", "user.name", "Practice Student")
    git_run(repo, "config", "--local", "user.email", "practice@example.invalid")


def passed(exercise: str) -> None:
    print(f"PASS: {exercise}")


git_version = subprocess.run(
    ["git", "--version"], check=True, text=True, capture_output=True
).stdout.strip()
print(git_version)
print("Safe practice helpers ready.")

In [ ]:
assert git_version.startswith("git version ")
assert ACTUAL_REPO == Path("/home/kanon/calgary-spatial-etl")
try:
    git_run(ACTUAL_REPO, "status")
except ValueError as error:
    assert "Refusing" in str(error)
else:
    raise AssertionError("The safety guard must reject the actual repository.")
passed("Safe setup")

# Part 1: Repository Map, Paths, and Ignore Policy

`pwd` means “print working directory.” Before any Git command, know whether your current path is the repository root. Git tracks paths relative to that root; directories are represented through tracked files, not as standalone objects.

For Calgary Spatial ETL, plan to track:

- `.env.example`, `.gitignore`, `README.md`, `docker-compose.yml`, `environment.yml`, `requirements.txt`
- `sql/`, `src/`, `tests/`, and `learning/`

Plan to ignore:

- `data/raw/*.geojson`, `data/processed/*.geojson`
- `outputs/logs/*.csv`, `outputs/qa/*.csv`, `.env`
- Python caches, notebook checkpoints, and editor state

Spatial data and generated outputs can be large, reproducible, frequently changing, or subject to licensing/privacy constraints. Source code and environment definitions should explain how to recreate them. `.gitignore` only prevents matching **untracked** files from being added; it does not untrack files already committed or staged.

In [ ]:
# Exercise 1: TODO - classify representative project paths.
track_paths = {
    ".env.example", ".gitignore", "README.md", "docker-compose.yml",
    "environment.yml", "requirements.txt", "sql/", "src/", "tests/", "learning/",
}
ignore_patterns = {
    "data/raw/*.geojson", "data/processed/*.geojson",
    "outputs/logs/*.csv", "outputs/qa/*.csv", ".env",
    "__pycache__/", "*.py[cod]", ".ipynb_checkpoints/", ".vscode/", ".idea/",
}
why_exclude_data = (
    "Raw, processed, and generated data can be large, reproducible, volatile, "
    "or restricted; version the code and instructions that recreate it."
)
gitignore_limit = ".gitignore does not untrack files that Git already tracks."

In [ ]:
assert {".env.example", ".gitignore", "src/", "tests/", "learning/"} <= track_paths
assert {".env", "data/raw/*.geojson", "outputs/qa/*.csv", ".ipynb_checkpoints/"} <= ignore_patterns
assert "reproducible" in why_exclude_data
assert "does not untrack" in gitignore_limit
passed("Exercise 1")

# Part 2: Initialize a Sandbox and Inspect `.git`

`git init` creates the hidden `.git` metadata directory containing the object database, references, configuration, and index state. The working files stay outside `.git`.

Git identity has three common scopes:

- **System:** machine-wide configuration.
- **Global:** your user account’s defaults.
- **Local:** one repository’s `.git/config`, overriding broader scopes.

This workbook never changes system or global identity. Each sandbox gets only `Practice Student <practice@example.invalid>` as local fake identity.

In [ ]:
# Exercise 2: TODO - initialize and configure a temporary repository.
practice_tempdir = TemporaryDirectory(prefix="git-practice-")
practice_repo = Path(practice_tempdir.name)
init_result = git_run(practice_repo, "init", "-b", "main")
configure_sandbox(practice_repo)

local_name = git_run(practice_repo, "config", "--local", "user.name").stdout.strip()
local_email = git_run(practice_repo, "config", "--local", "user.email").stdout.strip()
metadata_entries = {path.name for path in (practice_repo / ".git").iterdir()}
print(init_result.stdout.strip())
print(sorted(metadata_entries))

In [ ]:
assert (practice_repo / ".git").is_dir()
assert git_run(practice_repo, "branch", "--show-current").stdout.strip() == "main"
assert local_name == "Practice Student"
assert local_email == "practice@example.invalid"
assert {"config", "HEAD", "objects", "refs"} <= metadata_entries
passed("Exercise 2: deterministic sandbox init")

# Part 3: Working Tree, Index, and `HEAD`

Think of Git as three snapshots:

| Area | Meaning | Typical review command |
|---|---|---|
| Working tree | Files you are editing | `git diff` |
| Index (staging area) | Proposed next commit | `git diff --cached` or `git diff --staged` |
| `HEAD` | Last committed snapshot on the current branch | `git show HEAD` |

`git status --short` uses two columns: the first describes index state and the second working-tree state. `??` is untracked, `A ` is staged as added, ` M` is modified but unstaged, and `M ` is staged as modified. Stage explicit paths with `git add path`; avoid broad staging until you have reviewed what it would include.

In [ ]:
# Exercise 3: TODO - observe untracked, staged, then modified states.
readme = practice_repo / "README.md"
readme.write_text("# Sandbox Project\n", encoding="utf-8")
status_untracked = git_run(practice_repo, "status", "--short").stdout

git_run(practice_repo, "add", "README.md")  # Explicit path, intentionally chosen.
status_staged = git_run(practice_repo, "status", "--short").stdout

git_run(practice_repo, "commit", "-m", "Add sandbox README")
readme.write_text("# Sandbox Project\n\nPractice carefully.\n", encoding="utf-8")
status_modified = git_run(practice_repo, "status", "--short").stdout

print(status_untracked, status_staged, status_modified, sep="---\n")

In [ ]:
assert status_untracked == "?? README.md\n"
assert status_staged == "A  README.md\n"
assert status_modified == " M README.md\n"
assert git_run(practice_repo, "rev-list", "--count", "HEAD").stdout.strip() == "1"
passed("Exercise 3")

# Part 4: Review Diffs and Restore Deliberately

`git diff` compares working tree to index. After staging, that diff becomes empty because the working tree and index match; `git diff --cached` (same as `--staged`) then compares index to `HEAD`.

Two restore operations answer different questions:

- `git restore --staged README.md`: copy the `HEAD` version into the index, leaving your working edit intact.
- `git restore README.md`: copy the index version into the working tree, discarding the unstaged edit.

Always inspect status and the relevant diff first.

In [ ]:
# Exercise 4: TODO - compare, stage, unstage, and discard in the sandbox.
working_diff = git_run(practice_repo, "diff").stdout
git_run(practice_repo, "add", "README.md")
working_diff_after_add = git_run(practice_repo, "diff").stdout
cached_diff = git_run(practice_repo, "diff", "--cached").stdout
staged_diff = git_run(practice_repo, "diff", "--staged").stdout

git_run(practice_repo, "restore", "--staged", "README.md")
status_after_unstage = git_run(practice_repo, "status", "--short").stdout
git_run(practice_repo, "restore", "README.md")
status_after_restore = git_run(practice_repo, "status", "--short").stdout

In [ ]:
assert "+Practice carefully." in working_diff
assert working_diff_after_add == ""
assert cached_diff == staged_diff
assert "+Practice carefully." in cached_diff
assert status_after_unstage == " M README.md\n"
assert status_after_restore == ""
passed("Exercise 4")

# Part 5: Descriptive Commits and Inspection

A commit should capture one coherent change. Prefer an imperative subject that explains intent, such as `Document local setup`, over vague messages such as `updates`.

Use:

- `git status` before and after staging/committing.
- `git log --oneline` to scan history.
- `git show --stat HEAD` to inspect the latest commit and changed paths.
- `git show HEAD:path` to inspect a file exactly as committed.

In [ ]:
# Exercise 5: TODO - make one coherent change and inspect its commit.
(readme).write_text(
    "# Sandbox Project\n\nRun exercises only in temporary repositories.\n",
    encoding="utf-8",
)
git_run(practice_repo, "add", "README.md")
reviewed_staged_diff = git_run(practice_repo, "diff", "--cached").stdout
commit_result = git_run(practice_repo, "commit", "-m", "Document sandbox safety rule")
short_status = git_run(practice_repo, "status", "--short").stdout
one_line_log = git_run(practice_repo, "log", "--oneline", "-2").stdout
head_show = git_run(practice_repo, "show", "--stat", "--oneline", "HEAD").stdout
committed_readme = git_run(practice_repo, "show", "HEAD:README.md").stdout

In [ ]:
assert "+Run exercises only in temporary repositories." in reviewed_staged_diff
assert commit_result.returncode == 0
assert short_status == ""
assert "Document sandbox safety rule" in one_line_log
assert "README.md" in head_show
assert "temporary repositories" in committed_readme
passed("Exercise 5")

# Part 6: Debugging Drills

Debug Git by asking which area contains the unexpected state.

## Drill A: “Why is `output.csv` still shown?”

A pattern `generated/` ignores that directory at any depth. A pattern `/generated/` anchors it at the repository root. Use `git check-ignore -v path` to identify the matching rule.

## Drill B: “I added `.env` to `.gitignore`; why is it still tracked?”

Because ignore rules do not remove a path already in the index. In a real repository, first rotate any exposed secret and plan history remediation. Then, only after review, `git rm --cached .env` removes the path from future snapshots while leaving the working file. This workbook demonstrates only the ignore-matching diagnosis.

In [ ]:
# Debugging drill: TODO - repair the ignore rule and prove which pattern matches.
debug_tempdir = TemporaryDirectory(prefix="git-debug-")
debug_repo = Path(debug_tempdir.name)
git_run(debug_repo, "init", "-b", "main")
configure_sandbox(debug_repo)
(debug_repo / "generated").mkdir()
(debug_repo / "generated" / "output.csv").write_text("id,value\n1,10\n", encoding="utf-8")

# A mistaken learner rule might be "generated" for one file; use the directory rule instead.
(debug_repo / ".gitignore").write_text("generated/\n", encoding="utf-8")
ignore_check = git_run(
    debug_repo, "check-ignore", "-v", "generated/output.csv"
).stdout
status_with_ignore = git_run(debug_repo, "status", "--short").stdout
print(ignore_check)

In [ ]:
assert "generated/" in ignore_check
assert "generated/output.csv" in ignore_check
assert "output.csv" not in status_with_ignore
assert status_with_ignore == "?? .gitignore\n"
passed("Debugging drill")

# Part 7: Feature Branches, Switch, and Merge

A branch is a movable name pointing to a commit. `git switch -c feature/name` creates and checks out a branch; `git switch main` returns to `main`; `git merge feature/name` integrates the feature into the current branch.

Before switching, commit or deliberately handle current work. For this exercise, the feature adds a new file, so the merge is conflict-free and can fast-forward.

In [ ]:
# Exercise 6: TODO - create, commit on, and merge a feature branch.
git_run(practice_repo, "switch", "-c", "feature/add-notes")
(practice_repo / "NOTES.md").write_text("# Practice Notes\n", encoding="utf-8")
git_run(practice_repo, "add", "NOTES.md")
git_run(practice_repo, "commit", "-m", "Add practice notes")
feature_commit = git_run(practice_repo, "rev-parse", "HEAD").stdout.strip()

git_run(practice_repo, "switch", "main")
merge_result = git_run(practice_repo, "merge", "feature/add-notes")
main_commit = git_run(practice_repo, "rev-parse", "HEAD").stdout.strip()
branch_list = git_run(practice_repo, "branch", "--format=%(refname:short)").stdout.splitlines()

In [ ]:
assert merge_result.returncode == 0
assert main_commit == feature_commit
assert {"main", "feature/add-notes"} <= set(branch_list)
assert (practice_repo / "NOTES.md").read_text(encoding="utf-8") == "# Practice Notes\n"
assert git_run(practice_repo, "status", "--short").stdout == ""
passed("Exercise 6")

# Part 8: Remotes, GitHub, Authentication, and Upstream

A **remote** is a saved repository location; `origin` is the conventional name for the repository you cloned or intend to push to. For a local project, create an **empty** GitHub repository (no generated README, license, or `.gitignore`) to avoid unrelated initial history. Keep the local branch named `main`.

The first deliberate push is commonly `git push -u origin main`. `-u` records the upstream relationship so later `git pull` and `git push` know the default remote branch. These are concepts only here: this notebook never adds a remote or contacts GitHub.

Authenticate securely with the browser/OS credential manager (for example, Git Credential Manager) or an SSH key. Never paste a personal access token into a notebook, source file, command cell, or committed configuration.

Before publishing shared work, a common integration step is `git pull --rebase`: fetch remote commits and replay local commits on top, producing linear history. Stop and resolve conflicts carefully. Pull and push are network boundaries and must be run deliberately outside this notebook.

In [ ]:
# Exercise 7: TODO - explain remote vocabulary without executing network commands.
remote_concepts = {
    "origin": "Conventional local nickname for a remote repository URL.",
    "empty_github_repo": "A remote created without an initial README, license, or ignore file.",
    "main": "The intended primary branch name.",
    "upstream": "The remote branch tracked by a local branch.",
    "first_push": "git push -u origin main",
    "integrate": "git pull --rebase",
    "auth": "Browser/credential manager or SSH; never store notebook tokens.",
}
network_commands_are_concepts_only = True

In [ ]:
assert remote_concepts["first_push"] == "git push -u origin main"
assert remote_concepts["integrate"] == "git pull --rebase"
assert "upstream" in remote_concepts
assert "never" in remote_concepts["auth"]
assert network_commands_are_concepts_only is True
passed("Exercise 7")

# Part 9: Standard Workflow and Real-Repository Checklist

The repeatable collaboration loop is:

1. Edit one coherent change.
2. `git status --short`
3. `git diff`
4. `git add <explicit-paths>`
5. `git diff --cached`
6. `git commit -m "Describe the intent"`
7. `git pull --rebase`
8. `git push`

## Later manual practice in the actual Calgary repository

**Warning:** The commands below affect `/home/kanon/calgary-spatial-etl`. They are markdown only. Run them deliberately from the VS Code terminal after reviewing each command and its output. The repository currently has no commits, so there is no `HEAD` snapshot until the first commit.

```bash
cd /home/kanon/calgary-spatial-etl
pwd
git status --short
git branch --show-current
git diff
# Review the intended paths, then stage explicit files/directories:
git add .env.example .gitignore README.md docker-compose.yml environment.yml requirements.txt sql src tests learning
git diff --cached --name-only
git diff --cached
git commit -m "Initialize Calgary spatial ETL project"
# Only after creating and verifying an empty GitHub repository:
git remote add origin <YOUR-EMPTY-GITHUB-REPOSITORY-URL>
git push -u origin main
# On later collaborative cycles, after committing local work:
git pull --rebase
git push
```

Do not stage `.env`, ignored GeoJSON/CSV data, caches, checkpoints, or editor state. Confirm the staged names before committing.

# Capstone: Publish-Ready Local Commit, Without a Remote

In a fresh temporary repository:

1. Create `src/app.py`, `README.md`, `generated/output.csv`, and `.gitignore`.
2. Ignore `generated/`.
3. Stage only intended files with explicit paths.
4. Review staged names and staged diff.
5. Commit with a descriptive message.
6. Verify exactly one commit and a clean status.

No remote is added and no network command is run.

In [ ]:
# Capstone: TODO - rebuild this from the numbered prompt before checking.
capstone_tempdir = TemporaryDirectory(prefix="git-capstone-")
capstone_repo = Path(capstone_tempdir.name)
git_run(capstone_repo, "init", "-b", "main")
configure_sandbox(capstone_repo)

(capstone_repo / "src").mkdir()
(capstone_repo / "generated").mkdir()
(capstone_repo / "src" / "app.py").write_text(
    'print("Calgary spatial ETL")\n', encoding="utf-8"
)
(capstone_repo / "README.md").write_text("# Capstone Project\n", encoding="utf-8")
(capstone_repo / "generated" / "output.csv").write_text(
    "id,value\n1,42\n", encoding="utf-8"
)
(capstone_repo / ".gitignore").write_text("generated/\n", encoding="utf-8")

git_run(capstone_repo, "add", ".gitignore", "README.md", "src/app.py")
capstone_staged_names = git_run(
    capstone_repo, "diff", "--cached", "--name-only"
).stdout.splitlines()
capstone_staged_diff = git_run(capstone_repo, "diff", "--cached").stdout
git_run(capstone_repo, "commit", "-m", "Add capstone application scaffold")
capstone_commit_count = git_run(
    capstone_repo, "rev-list", "--count", "HEAD"
).stdout.strip()
capstone_status = git_run(capstone_repo, "status", "--short").stdout

In [ ]:
assert set(capstone_staged_names) == {".gitignore", "README.md", "src/app.py"}
assert "generated/output.csv" not in capstone_staged_names
assert "generated/output.csv" not in capstone_staged_diff
assert capstone_commit_count == "1"
assert capstone_status == ""
assert git_run(
    capstone_repo, "check-ignore", "generated/output.csv"
).stdout.strip() == "generated/output.csv"
assert git_run(capstone_repo, "remote").stdout == ""
passed("Capstone")

# Bridge Back to Calgary Spatial ETL

Without running Git against the real repository, explain the decisions you would make before its first commit.

Complete the dictionary in the next cell in your own words:

- What belongs in version control, and what belongs in `.gitignore`?
- What does staging mean?
- Which command reviews the proposed commit?
- What descriptive first-commit message would you use?
- Where is the boundary between local work and remote/network actions?

In [ ]:
# Bridge Back: TODO - replace each response with your own explanation.
bridge_back = {
    "track_and_ignore": (
        "Track code, tests, learning material, SQL, docs, and environment definitions; "
        "ignore secrets, generated GeoJSON/CSV, caches, checkpoints, and editor state."
    ),
    "staging": "The index is the reviewed proposal for the next commit.",
    "review_command": "git diff --cached",
    "commit_message": "Initialize Calgary spatial ETL project",
    "remote_push_boundary": (
        "Commits are local; adding origin, pull --rebase, authentication, and push cross "
        "the remote/network boundary and must be deliberate terminal actions."
    ),
}

In [ ]:
assert set(bridge_back) == {
    "track_and_ignore", "staging", "review_command",
    "commit_message", "remote_push_boundary",
}
assert bridge_back["review_command"] in {"git diff --cached", "git diff --staged"}
assert "index" in bridge_back["staging"].lower()
assert "local" in bridge_back["remote_push_boundary"].lower()
assert "network" in bridge_back["remote_push_boundary"].lower()
passed("Bridge Back")

# Hidden Reference Solutions

Try each exercise from memory before opening these.

<details>
<summary>Exercise 1: track and ignore</summary>

Track reproducible source, tests, SQL, learning material, documentation, and environment definitions. Ignore `.env`, generated/raw spatial data, generated reports, Python caches, notebook checkpoints, and editor state. Remember that `.gitignore` does not untrack existing index entries.
</details>

<details>
<summary>Exercises 2-5: local repository cycle</summary>

Initialize with `git init -b main`; configure `user.name` and `user.email` with `--local`; create a file; inspect `status`; explicitly `add`; compare `diff` and `diff --cached`; commit descriptively; inspect with `log` and `show`. Use `restore --staged` to unstage and `restore` to discard an unstaged edit only after review.
</details>

<details>
<summary>Exercise 6: branch and merge</summary>

`git switch -c feature/add-notes`, edit and commit, `git switch main`, then `git merge feature/add-notes`. Confirm the expected files and clean status.
</details>

<details>
<summary>Exercise 7 and Bridge Back</summary>

`origin` is a conventional remote nickname; upstream connects the local branch to a remote branch. `git push -u origin main` establishes that relationship. Use credential-manager/browser auth or SSH, never notebook tokens. Review staged content with `git diff --cached`; treat pull/push as deliberate network actions.
</details>

In [ ]:
# Workbook self-audit: valid JSON metadata and Python syntax for every cell.
import ast
import json

notebook_path = Path(
    "/home/kanon/calgary-spatial-etl/learning/practice/git_version_control_practice.ipynb"
)
notebook_document = json.loads(notebook_path.read_text(encoding="utf-8"))
for cell_number, cell in enumerate(notebook_document["cells"], start=1):
    expected_language = "python" if cell["cell_type"] == "code" else "markdown"
    assert cell["metadata"]["language"] == expected_language, cell_number
    if cell["cell_type"] == "code":
        ast.parse("".join(cell["source"]), filename=f"cell-{cell_number}")

assert 32 <= len(notebook_document["cells"]) <= 38
passed(f"Workbook self-audit: {len(notebook_document['cells'])} cells")

# Review Schedule

Use retrieval, not rereading:

- **Today:** complete every TODO and assertion; explain working tree, index, and `HEAD` aloud.
- **Tomorrow:** redo Exercises 3-5 without opening the solutions.
- **In 3 days:** rebuild the capstone from its numbered prompt in a fresh kernel.
- **In 1 week:** write the standard workflow and Calgary track/ignore policy from memory.
- **In 2 weeks:** rehearse the manual first-commit checklist, stopping before remote commands.
- **Monthly:** explain secret remediation, destructive `reset --hard`, upstream tracking, and why `pull --rebase` precedes push in the collaboration workflow.

When finished, clean up live temporary directories with `practice_tempdir.cleanup()`, `debug_tempdir.cleanup()`, and `capstone_tempdir.cleanup()`, or restart the kernel. Never replace a sandbox path with the Calgary repository path.